# Topic 14 — K-Means Clustering

> **Goal of this notebook:** understand the most widely used clustering algorithm — partition data into `k` groups by iteratively moving centroids — including how to choose `k` with the elbow method and a from-scratch implementation of Lloyd's algorithm.

**Contents**
1. Concept & intuition
2. The mathematics (objective, Lloyd's algorithm, k-means++)
3. Choosing k & limitations
4. The data: synthetic blobs
5. Fitting K-Means & visualising clusters
6. The elbow method
7. K-Means from scratch (Lloyd's algorithm)
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

K-Means splits data into a chosen number `k` of clusters, each represented by its **centroid** (the mean of its points). It alternates two simple steps until stable:
1. **Assign** each point to the nearest centroid.
2. **Update** each centroid to the mean of its assigned points.

Repeat → centroids settle where they best summarise compact, roughly spherical groups. You must pick `k` in advance.

## 2. The Mathematics

K-Means minimises the **within-cluster sum of squares** (inertia) — total squared distance from points to their centroid:

$$J = \sum_{j=1}^{k} \sum_{x \in C_j} \lVert x - \mu_j \rVert^2$$

**Lloyd's algorithm** is coordinate descent on $J$: the assignment step fixes centroids and minimises over assignments; the update step fixes assignments and minimises over centroids (the mean is the minimiser). Each step lowers $J$, so it converges — but only to a **local** optimum, so initialisation matters.

**k-means++** spreads the initial centroids out (probability of picking a point grows with its distance from existing centroids), giving far better, more consistent results than random init. `n_init` runs it several times and keeps the best.

## 3. Choosing k & Limitations

- **Choosing k:** the **elbow method** plots inertia vs `k` and looks for the bend where extra clusters stop helping much. (Topic 17, silhouette analysis, is a more principled alternative.)
- **Limitations:** assumes **spherical, similarly sized** clusters; sensitive to scaling and outliers; struggles with elongated or nested shapes (DBSCAN handles those). Always **scale** features first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Data: Synthetic Blobs

Four well-separated 2D clusters — ideal for *seeing* what K-Means does.

In [ ]:
X, y_true = make_blobs(n_samples=600, centers=4, cluster_std=0.9, random_state=42)
plt.scatter(X[:, 0], X[:, 1], s=12, color='gray')
plt.title('Raw data (no labels)'); plt.show()

## 5. Fitting K-Means & Visualising Clusters

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = km.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=12)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, edgecolor='k', label='centroids')
plt.title(f'K-Means clusters (inertia={km.inertia_:.0f})')
plt.legend(); plt.show()

## 6. The Elbow Method

Inertia always drops as `k` rises; we want the `k` at the **elbow**, where the gain flattens.

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_ for k in ks]
plt.plot(list(ks), inertias, marker='o')
plt.axvline(4, color='r', ls='--', label='elbow at k=4')
plt.xlabel('k'); plt.ylabel('inertia (within-cluster SS)')
plt.title('Elbow method'); plt.legend(); plt.show()

## 7. K-Means From Scratch (Lloyd's Algorithm)

Assign → update → repeat. We compare the final inertia to scikit-learn's.

In [ ]:
def kmeans_scratch(X, k, n_iters=100, seed=42):
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(len(X), k, replace=False)]
    for _ in range(n_iters):
        # assign: nearest centroid
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = dists.argmin(axis=1)
        # update: mean of assigned points
        new = np.array([X[labels == j].mean(axis=0) if np.any(labels == j)
                        else centroids[j] for j in range(k)])
        if np.allclose(new, centroids):
            break
        centroids = new
    inertia = sum(((X[labels == j] - centroids[j]) ** 2).sum() for j in range(k))
    return labels, centroids, inertia

_, _, inertia_scratch = kmeans_scratch(X, k=4)
print('From-scratch inertia:', round(inertia_scratch, 1))
print('scikit-learn inertia:', round(km.inertia_, 1))

## 8. Pros, Cons & When to Use

**Pros**
- Simple, fast, and scales to large datasets.
- Easy to interpret (centroids = cluster prototypes).

**Cons**
- Must choose `k` up front.
- Assumes spherical, equal-sized clusters; sensitive to scale & outliers.
- Only finds a **local** optimum (mitigated by k-means++ and `n_init`).

**When to use**
- Large datasets with roughly globular, separable groups.
- Quick segmentation, vector quantisation, or as a preprocessing step.

## 9. Summary

- K-Means minimises **within-cluster variance** by alternating assign/update (Lloyd's algorithm).
- **k-means++** initialisation and multiple restarts avoid poor local optima.
- The **elbow method** suggested k=4 on the blobs, matching the data.
- Our from-scratch Lloyd's algorithm reached the same inertia as scikit-learn, confirming the mechanics.